# CNN From Scratch

Have you ever wondered how Neural Networks function and make decisions? Have you ever wanted to gain a deeper understanding of this complex technology? If so, this series of notebooks is for you! We'll be exploring the inner workings of Neural Networks and implementing them from scratch.

Our journey will start by constructing a basic Convolutional Neural Network (CNN) and then optimizing it to achieve improved performance. This hands-on approach will provide a practical understanding of the technology, allowing us to dive deeper into its intricacies.

Let's get started!

In [34]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools
import os
import plotly.express as px
import plotly.graph_objs as go
from IPython.display import Image, display


## Data Preprocessing

In [18]:
train = pd.read_csv('./input/train.csv')

train

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41996,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41997,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
41998,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [19]:
test = pd.read_csv('./input/test.csv')

test

,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
27996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
27997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
27998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [20]:
def one_hot_encode(
        indices,
        num_classes
):
    """
    Converts a list of integer indices into a one-hot encoding matrix

    Parameters:
    indices (list): a list of integer indices representing the classes
    num_classes (int): the number of classes

    Returns:
    one_hot (numpy.ndarray): a 2D numpy array of shape (len(indices), num_classes) containing the
    one-hot encoding representation
    """

    # Create a 2D numpy array of zeros with shape (len(indices), num_classes) and data type np.float32
    one_hot = np.zeros(
        (len(indices), num_classes),
        dtype=np.float32
    )

    # Set the elements at the specified indices to 1
    one_hot[np.arange(len(indices)), indices] = 1

    # Return the one-hot encoding matrix
    return one_hot

- iloc: integer-location based indexing (access data by row and column)

In [21]:
X = train.iloc[:,1:].values # set training data [rows:columns]

In [22]:
y = train.iloc[:,0].values # set training labels

In [23]:
X = X.reshape(-1,28,28,1) # reshape into a format that can be fed into our NN

X

array([[[[0],
         [0],
         [0],
         ...,
         [0],
         [0],
         [0]],

        [[0],
         [0],
         [0],
         ...,
         [0],
         [0],
         [0]],

        [[0],
         [0],
         [0],
         ...,
         [0],
         [0],
         [0]],

        ...,

        [[0],
         [0],
         [0],
         ...,
         [0],
         [0],
         [0]],

        [[0],
         [0],
         [0],
         ...,
         [0],
         [0],
         [0]],

        [[0],
         [0],
         [0],
         ...,
         [0],
         [0],
         [0]]],


       [[[0],
         [0],
         [0],
         ...,
         [0],
         [0],
         [0]],

        [[0],
         [0],
         [0],
         ...,
         [0],
         [0],
         [0]],

        [[0],
         [0],
         [0],
         ...,
         [0],
         [0],
         [0]],

        ...,

        [[0],
         [0],
         [0],
         ...,
         [0],


In [24]:
X = X / 255 # normalize X values between 0 and 1

Get the number of classes

In [29]:
train_lb = train.label.unique()

train_lb

array([1, 0, 4, 7, 3, 5, 8, 9, 2, 6])

In [27]:
n_classes = train.label.nunique()

n_classes

10

One Hot Encode y

In [28]:
y = one_hot_encode(y, n_classes)

y

array([[0., 1., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.]], shape=(42000, 10), dtype=float32)

Get subset of the data to train on

In [30]:
X_subset = X[:200]
y_subset = y[:200]

Verify the data

In [31]:
px.imshow(X_subset[20].reshape(28,28),color_continuous_scale='ice')

## Model Implemention

### > Padding

Padding involves adding a border of zeros around an image, in order to maintain its original shape after undergoing a convolution operation.

![](./images/image1.png)

In [32]:
def zero_pad(X, padding):
    """
    Padding with zeros all images of the dataset X

    Argument:
    X: numpy array of shape (m, n_Height, n_Width, n_C) representing a batch of m images
    padding (int): amount of padding around each image on vertical and horizontal dimensions

    Returns:
    X_pad: padded image of shape (m, n_Height + 2 * pad, n_Width + 2 * pad, n_c)
    """

    X_pad = np.pad(
        X,
        (
            (0,0),
            (padding, padding),
            (padding, padding),
            (0,0)
        ),
        'constant',
        constant_values=(0,0)
    )

    return X_pad

### > Forward Propagation

The forward pass equation for the 2D convolution layer in the given code can be represented as:
$$
Z = g(A_{prev}.W + b)
$$

where:
- $A_{prev}$: the output activations of the previous layer, with shape (batch_size,height_prev,width_prev,channels_prev)
- '.': represents the convolution operation
- $W$: weights with shape (filter_size,filter_size,channels_prev,filters)
- $b$: the biases with shape (1, 1, 1, filters)
- $g(x)$: activation function applied element-wise to the result of the convolution and bias addition
- $Z$: the output of the convolution layer, with shape (batch_size,height,width,filters)

We can calculate the dimensions of the output using the following equation:
$$
n_{H} = [\frac{n_{H_{prev}} - f}{stride}] + 1
$$
$$
n_{W} = [\frac{n_{W_{prev}} - f}{stride}] + 1
$$
$$
n_C = n_{C_{prev}}
$$

![](images/image2.png)

$$
output(i,j) = \sum_{m,n}{Image(i+m,j+n) \times Kernel(m,n)}
$$

example above:
$$
Kernel = \begin{bmatrix}
1 & 0 & 1 \\
0 & 1 & 0 \\
1 & 0 & 1
\end{bmatrix}
$$

$$
Image_1 = \begin{bmatrix}
1 & 1 & 1 \\
0 & 1 & 1 \\
0 & 0 & 1
\end{bmatrix}
$$

Hadamard product + sum:

$$
Kernel ⊙ Image_1 = \begin{bmatrix}
1 & 0 & 1 \\
0 & 1 & 0 \\
1 & 0 & 1
\end{bmatrix} ⊙ \begin{bmatrix}
1 & 1 & 1 \\
0 & 1 & 1 \\
0 & 0 & 1
\end{bmatrix} = \begin{bmatrix}
1 & 0 & 1 \\
0 & 1 & 0 \\
0 & 0 & 1
\end{bmatrix}
$$

$$\rightarrow 1 + 0 + 1 + 0 + 1 + 0 + 0 + 0 + 1 = 4$$

### > Back Propagation

Let's start by Implementing the backward pass for a CONV Layer.

- Computing $dA$:
This is the formula for computing $dA$ with respect to the cost for a certain filter $W_{c}$ and a given training example:
$$
dA += \sum_{h=0}^{n_H} \sum_{w=0}^{n_W} W_c \times dZ_{hw} \space{(1)}
$$

Where $W_c$ is a filter and $dZ_{hw}$ is a scaler corresponding to the gradient of the cost with respect to the output of the conv layer Z at the hth row and wth column (corresponding to the dot product taken at the ith stride left and jth stride down). Note that at each time, you multiply the same filter $W_c$ by a different $dZ$ when updating $dA$. We do so mainly because when computing the forward propagation, each filter is dotted and summed by a different a_slice. Therefore when computing the backprop for $dA$, you are just adding the gradients of all the a_slices.


- Computing $dW$:
This is formula for computing $dW_c$ ($dW_c$ is the derivative of one filter) with respect to the loss:
$$
dW_c += \sum_{h=0}^{n_H} \sum_{w=0}^{n_W} a_{slice} \times dZ_{hw} \space{(2)}
$$


- Computing $db$:
This is formula for computing $db$ with respect to the cost for a certain filter $W_c$:
$$
db = \sum_h \sum_w dZ_{hw} \space{(3)}
$$

### > Conv2D

In [ ]:
class Conv2D:
    """
    A 2D Convolutional Layer in a Neural Network.

    Attributes:
    - filters (int): The number of filters in the Convolutional Layer
    - filter_size (int): The size of the filters
    - input_channels (int, optional): The number of input channels. Default is 3
    - padding (int, optional): The number of zero padding to be added to the input image. Default is 0
    - stride (int, optional): The stride length. Default is 1
    - learning_rate (float, optional): The learning rate to be used during training. Default is 0.001
    - optimizer (object, optional): The optimization method to be used during training. Default is None
    - cache (dict, optional): A dictionary to store intermediate values during forward and backward pass. Default is None
    - initailized (bool, optional): A flag to keep track of whether the layer has been initialized. Default is False.
    """

    def __init__(
            self,
            filters,
            filter_size,
            input_channels=3,
            padding=0,
            stride=1,
            learning_rate=0.001,
            optimizer=None
    ):
        """
        Initialize the Conv2D layer with the given parameters
        Args:
        - filters (int): The number of filters in the Convolutional layer.
        - filter_size (int): The size of the filters.
        - input_channels (int, optional): The number of input channels. Default is 3.
        - padding (int, optional): The number of zero padding to be added to the input image. Default is 0.
        - stride (int, optional): The stride length. Default is 1.
        - learning_rate (float, optional): The learning rate to be used during training. Default is 0.001.
        - optimizer (object, optional): The optimization method to be used during training. Default is None.
        """
        self.filters = filters
        self.filter_size = filter_size
        self.input_channels = input_channels
        self.padding = padding
        self.stride = stride
        self.learning_rate = learning_rate
        self.optimizer = optimizer
        self.cache = None
        self.initialized = False


    def relu(self, Z):
        """
        Implement the ReLU function.

        Args:
        - Z: Output of the linear layer

        Returns:
        - A: Post activation parameter
        - cache: used for backpropagation
        """

        A = np.maximum(0, Z)
        cache = Z

        return A, cache
    

    def relu_backward(self, dA, activation_cache):
        """
        Implement the backward propagation for a single ReLU unit.

        Args:
        - dA: post activation gradient, of any shape
        - activation_cache: Z where we store for computing backward propagation efficiently

        Returns:
        - dZ: Gradient of the cost with respect to Z
        """

        Z = activation_cache
        dZ = np.array(dA, copy=True) # just converting dz to a correct object

        dZ[Z <= 0] = 0 # when z <= 0, you should set dz to 0 as well

        return dZ
    

    def conv_single_step(self, a_slice_prev, W, b):
        """
        Apply one filter defined by parameters W on a single slice (a_slice_prev) of the output
        activation of the previous layer.

        Parameters:
        - a_slice_prev: slice of input data of shape (f, f, n_C_prev)
        - W: weight parameters contained in a window - matrix of shape (f, f, n_C_prev)
        - b: bias parameters contained in a window - matrix of shape (1, 1, 1)

        Returns:
        - A: result of applying the activation function to Z
        - Cache: used for backpropagation
        """

        s = np.multiply(a_slice_prev, W)
        Z = np.sum(s)
        Z = Z + float(b)

        return Z
    

    def forward(self, A_prev):
        """
        Implements the forward propagation for a convolution function

        Parameters:
        - A_prev: output activations of the previous layer, numpy array of shape (m, n_H_prev, n_W_prev, n_C_prev)

        Returns:
        - Z: conv output, numpy array of shape (m, n_H, n_W, n_C)
        - Cache: cache of values needed for the conv_backward() function
        """

        # Create list to store activation cache for backprop
        activation_caches = []

        # Initialize neural network
        if self.initialized == False:
            np.random.seed(0)
            # W
            self.W = np.random.randn(
                self.filter_size,
                self.filter_size,
                A_prev.shape[-1],
                self.filters
            )
            # b
            self.b = np.random.randn(
                1,
                1,
                1,
                self.filters
            )
            self.initialized = True

        # Retrieve dimensions from A_prev's shape
        (m, n_H_prev, n_W_prev, n_C_prev) = A_prev.shape

        # Retrieve dimensions from W's shape
        (f, f, n_C_prev, n_C) = self.W.shape

        # Compute the dimensions of the output volume
        n_H = int((n_H_prev - f + (2 * self.padding)) / self.stride) + 1
        n_W = int((n_W_prev - f + (2 * self.padding)) / self.stride) + 1

        # Initialize the output volume Z with zeros
        Z = np.zeros((m, n_H, n_W, n_C))

        # Add padding to A_prev
        A_prev_pad = zero_pad(A_prev, self.padding)

        # Loop over the batch of training examples
        for i in range(m):
            # Select ith training example's padded activation
            a_prev_pad = A_prev_pad[i]
            for h in range(n_H):
                # Find the vertical start and end
                vert_start = h * self.stride
                vert_end = vert_start + f

                # Loop over horizontal axis of the output volume
                for w in range(n_W):
                    # Find the horizontal start
                    horiz_start = w * self.stride
                    horiz_end = horiz_start + f

                    # Loop over channels
                    for c in range(n_C):
                        # Use the corners to define the slice of a_prev_pad
                        a_slice_prev = a_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :]
                        # Convolve the slice with the filter W and bias b
                        weights = self.W[:, :, :, c]
                        biases = self.b[:, :, :, c]
                        Z[i, h, w, c] = self.conv_single_step(a_slice_prev, weights, biases)

            # Apply ReLU activation and store cache for backpropagation
            Z[i], activation_cache = self.relu(Z[i])
            # Append the activation to the caches list
            activation_caches.append(activation_cache)

        self.cache = (A_prev, np.array(activation_caches))

        return Z
    


    def backward(self, dZ):
        """
        Implement the backward propagation for a convolution function

        Parameters:
        - dZ: gradient of the ost with respect to the output of the conv layer (Z), numpy array
        of shape (m, n_H, n_W, n_C)
        - Cache: cache of values needed for conv_backward(), output of conv_forward()

        Returns:
        - dA_prev: gradient of the cost with respect to the input of the conv layer (A_prev),
        numpy array of shape (m, n_H_prev, n_W_prev, n_C_prev)
        - dW: gradient of the cost with respect to the weights of the conv layer (W),
        numpy array of shape (f, f, n_C_prev, n_C)
        - db: gradient of the cost with respect to the biases of the conv layer (b),
        numpy array of shape (1, 1, 1, n_C) 
        """

        # Retrieve information from "cache"
        A_prev, activation_cache = self.cache
        W, b = self.W, self.b

        # Retrieve dimensions from A_prev's shape
        (m, n_H_prev, n_W_prev, n_C_prev) = A_prev.shape

        # Retrieve dimensions from W's shape
        (f, f, n_C_prev, n_C) = W.shape

        # Retrieve strides, padding information
        stride = self.stride
        pad = self.padding

        # Retrieve dimensions from dZ's shape
        (m, n_H, n_W, n_C) = dZ.shape

        # Initialize dA_prev, dW, db
        dA_prev = np.zeros((m, n_H_prev, n_W))
        self.dW = np.zeros((f, f, n_C_prev, n_C))
        self.db = np.zeros((1, 1, 1, n_C))

        # Pad A_prev and dA_prev
        A_prev_pad = zero_pad(A_prev, pad)
        dA_prev_pad = zero_pad(dA_prev, pad)

        # Loop Over the training examples
        for i in range(m):
            # Compute gradients of the activation function
            dZ[i] = self.relu_backward(dZ[i], activation_cache[i])
            # Select ith training example from A_prev_pad and dA_prev_pad
            a_prev_pad = A_prev_pad[i]
            da_prev_pad = dA_prev_pad[i]

            # Loop over vertical axis of the output volume
            for h in range(n_H):
                vert_start = h * stride
                vert_end = vert_start + f

                # Loop over horizontal axis of the output volume
                for w in range(n_W):
                    horiz_start = w * stride
                    horiz_end = horiz_start + f

                    # Loop over the channels of the output volume
                    for c in range(n_C):
                        # Find a slice using the dimensions
                        a_slice = a_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :]
                        # Update gradients for the window and the filter's parameters
                        da_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :] += W[:, :, :, c] * dZ[i, h, w, c]
                        self.dW[:, :, :, c] += a_slice * dZ[i, h, w, c]
                        self.db[:, :, :, c] += dZ[i, h, w, c]

            # Set the ith training example's dA_prev to the unpaded da_prev_pad
            if pad:
                dA_prev[i, :, :, :] = da_prev_pad[pad:-pad, pad:-pad, :]
            else:
                dA_prev[i, :, :, :] = dA_prev[i, :, :, :]

        self.update_parameters(self.optimizer)
        return dA_prev


    def Adam(self, beta1=0.9, beta2=0.999):
        """
        """


    
    def update_parameters(self, optimizer=None):
        """
        """